<a href="https://colab.research.google.com/github/H-Mike-Yankee/flyrank-ml-internship-starter/blob/main/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane chosen: Lane 2 — Refresh / Content Opportunity Scoring**

I'm choosing this lane over freestyle and over the other three predefined lanes for three reasons.

**Fit with my background.** My prior projects (a disease-prediction system and a nutrition
recommendation API) follow the same underlying pattern: train or integrate a model, then use
its output to support a real decision someone acts on. Lane 2 asks the same kind of question —
"which pages should be reviewed first?" — so I can bring an existing way of thinking about
model-to-decision pipelines rather than starting from zero.

**The lane has a working reference to learn from, not just a data dump.** The starter repo
already ships a full pipeline for this exact lane — feature prep, a transparent baseline score,
a trained model, and evaluation (scripts/01–05). That means my job in the early weeks isn't
to invent a methodology from nothing; it's to understand why this pipeline is built the way
it is, run it honestly on the data, and then find where I can improve or extend it.

**The decision, the actor, and the cost are already concrete.** Unlike a purely exploratory
lane (Lane 1) or a descriptive one (Lane 3), Lane 2 forces me to name an actual decision
(which pages to review first), an actual actor (a content reviewer with limited time), and an
actual cost of getting it wrong (wasted review time on a page that wasn't really a problem,
while a genuinely declining page goes unnoticed).

I'm treating this as provisional, not final — the guide allows changing lanes through week 4.

## 2. The question: decision, action, cost of a wrong call

**Decision:** Out of thousands of content pages, which ones should a content reviewer
look at first this week, given they only have capacity to review a limited number
(e.g. the top 50)?

**Unit of analysis:** One page (`content_id`).

**Who acts on it:** A content/SEO reviewer with limited weekly review capacity — they
cannot manually check all 30,000 pages, so they need a ranked shortlist of the pages
most worth their time.

**The action:** The reviewer opens the flagged page, checks the reason code(s) attached
to it (e.g. stale but still getting impressions, declining despite demand, thin content
with traffic, good position but aging), and decides whether to refresh, expand, or
otherwise update it.

**Cost of a wrong call:** There are two ways this can go wrong, and they are not equally
cheap. A false positive — flagging a page that didn't actually need attention — wastes
a limited and valuable resource: the reviewer's time. A false negative — missing a page
that is genuinely declining — is more costly, because that page's traffic loss goes
unnoticed until it has already dropped significantly, and by the time someone notices
through normal monitoring, meaningful traffic (and the business value behind it) may
already be gone. Because reviewer time is the scarcer resource in this system, and
because undetected decline compounds silently, I treat missed declining pages (false
negatives) as the more expensive error of the two.

**Why data/ML helps here:** A simple rule (e.g. "flag every page older than 6 months")
would either flag far too many pages to be useful, or miss the ones where decline is
real but doesn't fit one simple pattern. The signals that actually indicate risk
(traffic trend, position, content age, CTR, engagement) interact in ways that are too
tangled to capture with a handful of hand-written rules — which is exactly the gap the
starter pipeline's baseline-vs-model comparison is designed to test.

## 3. Quick look at the data (2-3 real numbers)



In [ ]:
import pandas as pd

# Load the starter dataset using the exact confirmed path
df = pd.read_csv("/content/flyrank-ml-internship/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv")

# 1. Basic shape check
print(f"Total pages: {len(df):,}")
print(f"Total clients: {df['client_id'].nunique()}")

# 2. How many pages are flagged as declining (our label source)?
declining_count = (df['trend_direction'] == 'down').sum()
declining_pct = declining_count / len(df) * 100
print(f"Pages currently declining (trend_direction == 'down'): {declining_count:,} ({declining_pct:.1f}%)")

# 3. How many pages have real reviewer-relevant volume (i.e. worth reviewing at all)?
visible_pages = (df['impressions_90d'] >= 500).sum()
print(f"Pages with 500+ impressions in 90 days (real visibility): {visible_pages:,}")

Total pages: 30,000
Total clients: 32
Pages currently declining (trend_direction == 'down'): 16,262 (54.2%)
Pages with 500+ impressions in 90 days (real visibility): 16,726


**What these numbers show:**

Out of 30,000 pages across 32 clients, **16,262 pages (54.2%) are currently flagged as
declining** by the simple trend rule (`trend_direction == "down"`). That is a large enough
group that a reviewer genuinely cannot check them all by hand — this is exactly the scale
problem Lane 2 is meant to solve.

At the same time, **16,726 pages (55.8%) have real visibility** (500+ impressions in 90
days) — meaning most of the pages in this dataset are not just noise or abandoned content;
they are pages that matter, with real search traffic behind them.

Together, these numbers show the shape of the problem: a majority-declining, majority-visible
dataset, where a reviewer with limited capacity needs a way to prioritize — which is exactly
why a ranked, evidence-based review queue (rather than a simple age-based rule) is worth
building over the next 7 weeks.

## 4. Careful words: what I can and can't claim

**What this work CAN say:**
- Which pages, based on observed signals (traffic trend, position, content age, CTR,
  engagement), are most worth a reviewer's limited time this week.
- That a learned ranking, tested honestly against held-out data, performs better or worse
  than a simple rule-based baseline at picking pages worth reviewing.
- Directional patterns: for example, that pages with certain combinations of signals
  (e.g. high impressions + low CTR) are over-represented among currently declining pages.
- Decision-support: a ranked list with reason codes a human reviewer can inspect, question,
  and override — not a final verdict.

**What this work CANNOT say:**
- That refreshing a page will cause its traffic to recover — that would require an actual
  experiment (e.g. before/after with a control group), which this data does not provide.
- That any result here reveals or predicts Google's search ranking algorithm.
- That a page is "declining because of X" in a causal sense — the data shows association,
  not cause. A page could be losing traffic due to seasonality, consolidation into a sibling
  page, or a SERP change, none of which this dataset can fully rule out on its own.
- That AI-referral patterns say anything about AI citations or AI search rankings — the
  data only measures sessions where someone clicked through from an AI tool, nothing more.

All findings in this work will be framed as **observed**, **measured**, or **directional**,
and outputs will be framed as **decision-support** for a human reviewer — never as proof,
prediction, or a guaranteed outcome.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs
it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.